In [2]:
#!/usr/bin/env python3
"""
test_model.py
=============
Live webcam test of your trained sign model -- the milestone to clear BEFORE
Flutter. It runs the exact same pipeline the phone will run:

    raw webcam -> hand landmarks -> position-ordered 126 features ->
    wrist-center + scale normalize -> 30-frame buffer -> sign_model.tflite -> word

To guarantee there's no mismatch, it IMPORTS the real functions from
collect_data.py (feature extraction) and preprocess.py (normalization) instead
of re-implementing them. If it recognizes your signs here, the phone just has to
match these conventions -- which sign_classifier.dart already does.

Run after train.py:
    python test_model.py

Controls:  q = quit
"""

import json
from collections import deque
from pathlib import Path

import cv2
import numpy as np
import mediapipe as mp
import tensorflow as tf

# Reuse the SAME logic as the rest of the pipeline (no drift!)
from collect_data import (ensure_model, make_detector, draw_landmarks,
                          extract_features, FRAMES_PER_SEQUENCE, CAM_INDEX)
from preprocess import normalize_frame

TFLITE = Path("sign_model.tflite")
LABELS = Path("labels.json")
DATASET = Path("dataset.npz")
CONF_THRESHOLD = 0.85          # turns the label green when the model is sure


def load_interpreter():
    interp = tf.lite.Interpreter(model_path=str(TFLITE))
    inp = interp.get_input_details()[0]
    interp.resize_tensor_input(inp["index"], [1, FRAMES_PER_SEQUENCE, 126])
    interp.allocate_tensors()
    return interp


def predict(interp, seq):
    """seq: (30, 126) float32 -> probability vector."""
    inp = interp.get_input_details()[0]
    out = interp.get_output_details()[0]
    interp.set_tensor(inp["index"], seq[None, ...].astype(np.float32))
    interp.invoke()
    return interp.get_tensor(out["index"])[0]


def offline_eval(interp):
    """Quick numeric sanity check on the held-out test set, if available."""
    if not DATASET.exists():
        return
    d = np.load(DATASET)
    X, y = d["X_test"], d["y_test"]
    correct = sum(int(np.argmax(predict(interp, xi))) == int(yi)
                  for xi, yi in zip(X, y))
    print(f"Offline TFLite test accuracy: {correct/len(X)*100:.1f}% "
          f"on {len(X)} held-out samples")


def overlay(frame, word, conf, fill):
    h, w = frame.shape[:2]
    strip = frame.copy()
    cv2.rectangle(strip, (0, h - 90), (w, h), (0, 0, 0), -1)
    cv2.addWeighted(strip, 0.55, frame, 0.45, 0, frame)
    color = (0, 220, 0) if conf >= CONF_THRESHOLD else (200, 200, 200)
    cv2.putText(frame, word, (16, h - 38),
                cv2.FONT_HERSHEY_SIMPLEX, 1.4, color, 3, cv2.LINE_AA)
    cv2.putText(frame, f"{conf*100:4.0f}%", (w - 130, h - 38),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2, cv2.LINE_AA)
    # buffer-fill bar
    bw = int((w - 32) * fill)
    cv2.rectangle(frame, (16, h - 20), (16 + bw, h - 12), (80, 160, 255), -1)


def main():
    if not TFLITE.exists():
        raise SystemExit("sign_model.tflite not found. Run train.py first.")
    ensure_model()
    labels = json.loads(LABELS.read_text())
    interp = load_interpreter()
    offline_eval(interp)
    print("Labels:", labels)

    detector = make_detector()
    cap = cv2.VideoCapture(CAM_INDEX)
    if not cap.isOpened():
        raise SystemExit(f"Could not open camera index {CAM_INDEX}.")

    buf = deque(maxlen=FRAMES_PER_SEQUENCE)
    ts = 0
    word, conf = "...", 0.0

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        # raw frame (no flip) -- matches collect_data.py and the phone
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        ts += 33
        result = detector.detect_for_video(mp_img, ts)

        feats = normalize_frame(extract_features(result))  # SAME as training
        buf.append(feats)
        draw_landmarks(frame, result)

        if len(buf) == FRAMES_PER_SEQUENCE:
            probs = predict(interp, np.array(buf, dtype=np.float32))
            top = int(np.argmax(probs))
            word, conf = labels[top], float(probs[top])

        overlay(frame, word, conf, len(buf) / FRAMES_PER_SEQUENCE)
        cv2.imshow("test_model  (q to quit)", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

Done.
Offline TFLite test accuracy: 100.0% on 47 held-out samples
Labels: ['hello', 'help', 'iloveyou', 'no', 'please', 'sorry', 'thanks', 'yes']


c:\Users\aliak\AppData\Local\Programs\Python\Python310\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
